In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

In [ ]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

.

.

## Load the `SelectedFeature` Dataset
* Result from the DA3-2

In [ ]:
FeatureSelected = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureSelected.csv', header=None)
FeatureSelected.shape

#### Standardize Features

- Don't forget to transpose `FeatureSelected` before standardizing.
  - First dimension of dataset should be the **"number of data samples"**.


In [ ]:
# Standardize feature values
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

FeatureSelected_T = FeatureSelected.T
FeatureSelected_std = StandardScaler().fit_transform(FeatureSelected_T)
FeatureSelected_std.shape

## Preparing Training, Validation, and Test Data

To evaluate the performance of an unsupervised model (Autoencoder, AE), we only need to split the NORMAL dataset into two parts:  
- **Training set (80% of Nomral)**: used to train the AE model.  
- **Validation set (20% of Normal)**: used to evaluate the AE model’s reconstruction performance (NOT fault detection performance).  

In addition, we also need TEST data set for the unsupervised fault detection.
- **Test set (100% of Normal & Abnormal)**: used to evaluate AE model's fault detection performance.

Note that we don't need labels for AE training.


In [ ]:
NormalSet   = FeatureSelected_std[:180 , :]
AbnormalSet = FeatureSelected_std[180: , :]

NormalSet.shape, AbnormalSet.shape

In [ ]:
from sklearn.model_selection    import train_test_split

TestData_Ratio = 0.2
TrainData_Nor, ValidData_Nor = train_test_split(NormalSet  , test_size=TestData_Ratio, random_state=777)

print(TrainData_Nor.shape, ValidData_Nor.shape)

In [ ]:
TrainData  = TrainData_Nor
ValidData  = ValidData_Nor
TestData   = np.concatenate([NormalSet  , AbnormalSet], axis=0)

print(TrainData.shape, ValidData_Nor.shape, TestData.shape)

.

.

.

.

.

## Training Autoencoder (AE)


- **Input Layer**: matches the number of input features in the dataset.  
- **Encoding/Decoding Layers**: use `ReLU` activation to capture non-linear relationship in the data.  
- **Output Layer**: uses `linear` (NOT `softmax`) activation with same number of nodes as theinput layer, to reconstruct the original data.



In [ ]:
def AE_model(input_data):
    keras.backend.clear_session() # clearing the Keras backend session (initiating variables)

    model = keras.Sequential()
    model.add(keras.layers.InputLayer(input_shape = (input_data.shape[1],) ))                               # Input Layer
    model.add(keras.layers.Dense(units = 15, activation = keras.activations.relu, name = 'Encoder_1'))      # Encoder
    model.add(keras.layers.Dense(units = 10, activation = keras.activations.relu, name = 'Encoder_2'))      # Encoder
    model.add(keras.layers.Dense(units = 5 , activation = keras.activations.relu, name = 'LatentSpace'))    # Bottle neck
    model.add(keras.layers.Dense(units = 10, activation = keras.activations.relu, name = 'Decoder_1'))      # Decoder
    model.add(keras.layers.Dense(units = 15, activation = keras.activations.relu, name = 'Decoder_2'))      # Decoder
    model.add(keras.layers.Dense(units = input_data.shape[1], activation = keras.activations.linear, name = 'Output')) # Output (Reconstruction)

    model.compile(optimizer = keras.optimizers.Adam(learning_rate = 0.001), # Optimizer
                  loss = keras.losses.MeanSquaredError())                   # Loss for reconstruction error between input and output

    return model

In [ ]:
# Check the model architecture and the number of parameters
AE = AE_model(TrainData)
AE.summary()

In [ ]:
# Callback 1: CheckProcess
EpochForPrint = 20
class CheckProcess(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        keras.callbacks.Callback()
        if epoch%EpochForPrint == 0:
            print("[{} Epochs] Validation loss: {:.4f}  ".format(epoch, logs['val_loss']))

In [ ]:
# Callback 2: Early Stopping
EalryStop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=40, restore_best_weights=True)

In [ ]:
# Model traning and validation
TraingHistory = AE.fit(TrainData, TrainData,
                       validation_data=(ValidData, ValidData),
                       epochs=1000, verbose=0,
                       callbacks=[EalryStop, CheckProcess()])

In [ ]:
ReconstructedFeature = AE.predict(TestData)
ReconstructedFeature.shape

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(TraingHistory.history['loss'], label='Training loss', c = 'tab:blue')
plt.plot(TraingHistory.history['val_loss'], label='Validation loss', c = 'tab:orange')
plt.xlabel('epoch', fontsize=15)
plt.ylabel('loss', fontsize=15)
plt.legend(fontsize=15)
plt.grid(alpha=0.5)
plt.show()

.

.

## Comparing Original vs. Reconstructed Feature

In [ ]:
DataIndex = 0 # Index 0~179: Normal / 180~359: Abnormal

plt.figure(figsize=(8, 5))
plt.plot(np.arange(30),             TestData[DataIndex], 'o-', label='Original')
plt.plot(np.arange(30), ReconstructedFeature[DataIndex], 'o-', label='Reconstructed')
plt.title('Original vs. Reconstruction', fontsize=15)
plt.xlabel('Feature Index')
plt.ylabel('Feature Value')
plt.legend()
plt.grid(alpha=0.5)
plt.show()

## Confirming Reconstruction Error

In [ ]:
# Define a function calculating the Mean Squared Errors (MSEs)
def recon_error(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean((X - X_pred)**2, axis=1)

err_train = recon_error(AE, TrainData)      # MSEs of training data (normal)
err_valid = recon_error(AE, ValidData)      # MSEs of validation data (normal)
err_test  = recon_error(AE, TestData )      # MSEs of test data (nor + abn)
err_testN = recon_error(AE, TestData[:180]) # MSEs of test data (only normal)
err_testA = recon_error(AE, TestData[180:]) # MSEs of test data (only abonormal)

print("TrainData Error: {:.2f}\n".format(err_train.mean()) +
      "ValidData Error: {:.2f}\n\n".format(err_valid.mean()) +
      " Test_Tot Error: {:.2f}\n".format(err_test.mean()) +
      " Test_Nor Error: {:.2f}\n".format(err_testN.mean()) +
      " Test_Abn Error: {:.2f}".format(err_testA.mean()))

In [ ]:
AE.save('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/AE_model.keras')

.

.

.

## Fault Detection Using Autoencoder (AE)

Using the AE trained **only on normal data** (unsupervised learning), we can detect faulty or abnormal conditions.

- When an abnormal sample is input to the AE, the **reconstruction error** is expected to be much larger than when a normal sample is input.  
  This is because the AE has learned to accurately reconstruct only the normal patterns.

- Therefore, we can set a **threshold** on the reconstruction error to decide whether a new sample is normal or abnormal.

---

### 📈 Two Representative Methods for Determining the Threshold

1. **Statistical method (mean + 3σ rule)**  
   - Use the mean and standard deviation of the reconstruction errors obtained from **normal data only**,  
   and set the threshold as:   $\text{Threshold} = \mu + 3\sigma$

   - Samples with reconstruction error greater than this threshold are classified as *abnormal*.

2. **ROC (Receiver Operating Characteristic) curve-based method**  
   - Evaluate different threshold values and measure the corresponding **True Positive Rate (TPR)** and **False Positive Rate (FPR)**.  
   - The optimal threshold can then be selected using a criterion such as **Youden’s J statistic (TPR − FPR)**.

.

- The **mean + 3σ** rule is a simple unsupervised approach,  
  while **ROC-based analysis** provides a more data-driven and balanced threshold when labels are available.

### **Step 1. Prepare Test Labels**
We first define the ground truth labels for the test dataset.  
- Normal data (first 180 samples): labeled as **0**  
- Abnormal (fault) data (next 180 samples): labeled as **1**  

In [ ]:
TestLabel = np.zeros(TestData.shape[0])
TestLabel[180:] = 1
TestLabel

### **Step 2. Set *Threshold 1* using Mean + 3σ Rule**
To determine a decision boundary for abnormal detection:  
- Compute the **mean (μ)** and **standard deviation (σ)** of reconstruction errors from normal data.  
- Define the threshold as:  

  $\text{Threshold} = \mu + 3\sigma$
  
- Samples whose reconstruction error exceeds this threshold are predicted as **abnormal (1)**.

In [ ]:
mu, sigma = err_testN.mean(), err_testN.std()
Threshold_3sigma = mu + 3*sigma
Predict_3sigma = np.where(err_test > Threshold_3sigma, 1, 0)
Predict_3sigma

### **Step 3. Set *Threshold 2* by evaluating ROC Curve and AUC**
To analyze the AE’s discrimination ability:  
- Compute **TPR (True Positive Rate)** and **FPR (False Positive Rate)** for different thresholds using the `roc_curve()` function.  
- Plot the **ROC curve** and calculate **AUC (Area Under Curve)** to measure how well the AE separates normal and abnormal samples.  
  - AUC close to 1.0 indicates excellent separation performance.  
- The diagonal gray line represents random guessing (AUC = 0.5).

In [ ]:
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix

# ROC/AUC
fpr, tpr, ths_roc = roc_curve(TestLabel, err_test)
roc_auc = auc(fpr, tpr)
print(f"AUC (ROC): {roc_auc:.3f}\n")

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f"ROC Curve (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], color='gray', linestyle='--', lw=1)  # random guess line
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve for AE-based Anomaly Detection')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

Instead of a fixed 3σ rule, we can choose a **data-driven threshold** from the ROC curve.  
- Compute **Youden’s J statistic** as \( J = TPR - FPR \).  
- The threshold corresponding to the **maximum J** gives the best trade-off between detection rate and false alarms.  
- This threshold (`Threshold_ROC`) is then used to classify samples as normal or abnormal.

In [ ]:
j_scores      = tpr - fpr
max_index     = np.argmax(j_scores)
Threshold_ROC = ths_roc[max_index]

print(f"Maximum_Index: {max_index}\n" +
      f"Threshold_ROC: {Threshold_ROC:.3f}\n")

Predict_ROC = np.where(err_test > Threshold_ROC, 1, 0)
Predict_ROC

### **Step 5. Compare Confusion Matrices**
Visualize and compare confusion matrices for:
- **3-Sigma Threshold**  
- **ROC-based Threshold**  

The confusion matrix shows:
- True Normal (TN)
- False Alarm (FP)
- Missed Fault (FN)
- Correct Fault Detection (TP)

A darker diagonal pattern indicates better classification performance.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm1 = confusion_matrix(TestLabel, Predict_3sigma)

plt.figure(figsize=(4, 4))
sns.heatmap(cm1, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix using 3-Sigma Threshold")
plt.show()

print("\n")

cm2 = confusion_matrix(TestLabel, Predict_ROC)

plt.figure(figsize=(4, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix using ROC Threshold")
plt.show()

### **Step 6. Evaluation Metrics Comparison**
Quantitatively compare the two thresholding methods using the following metrics:
- **Accuracy:** overall correct predictions  
- **Precision:** ratio of correctly detected faults among predicted faults  
- **Recall:** ratio of correctly detected faults among actual faults  
- **F1-Score:** harmonic mean of Precision and Recall  

A summary table (`EvalTable`) helps identify which method (3σ or ROC) provides better overall fault detection performance.

In [ ]:
from sklearn import metrics

# Calculate the evaluation metrics
accuracy_1  = metrics.accuracy_score( TestLabel, Predict_3sigma)
precision_1 = metrics.precision_score(TestLabel, Predict_3sigma)
recall_1    = metrics.recall_score(   TestLabel, Predict_3sigma)
f1_score_1  = metrics.f1_score(       TestLabel, Predict_3sigma)

accuracy_2  = metrics.accuracy_score( TestLabel, Predict_ROC)
precision_2 = metrics.precision_score(TestLabel, Predict_ROC)
recall_2    = metrics.recall_score(   TestLabel, Predict_ROC)
f1_score_2  = metrics.f1_score(       TestLabel, Predict_ROC)


EvalTable = pd.DataFrame(columns=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                         index=['3-Sigma', 'ROC'],
                         data=[[accuracy_1, precision_1, recall_1, f1_score_1],
                               [accuracy_2, precision_2, recall_2, f1_score_2]])

EvalTable